In [1]:
import pandas as pd
import numpy as np
import torch
from  torch.optim import AdamW, Adam, SGD, RMSprop
from transformers import DistilBertTokenizerFast, DistilBertForSequenceClassification, Trainer, TrainingArguments, get_linear_schedule_with_warmup
from datasets import Dataset, DatasetDict
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import gc
from transformers import EarlyStoppingCallback
from utils.config import *

In [2]:
dataset_path= 'Model_dataset/synthetic_question_ans_data-v2.csv'

# q type classifer,
q_type_model_name= 'distilbert-base-uncased'
q_type_model_result= '.temp/model_results/q_types_model_lite_results'
q_type_model= '.temp/model/fine_tuned_question_classifier_model_lite'



# Question type classsifier

### Preprocessing

In [ ]:
df= pd.read_csv(dataset_path)
df= df[["question", "question_type"]]
df.info()

In [ ]:
df["question_type"].unique()

In [5]:
df.drop_duplicates(inplace= True)

In [ ]:
# adding labels
label_mapping = {key: index for index, key in enumerate(QUESTION_TYPES)}
df['label'] = df['question_type'].map(label_mapping)

# droping unused column
df.drop('question_type', axis=1,  inplace= True)
df.head()

In [ ]:
df= df.sample(frac=1).reset_index(drop=True)
df.info()

In [ ]:
df.sample(5)

### Retraing Preparations:

In [9]:
#convert to hugging face dataset
dataset= Dataset.from_pandas(df)

#Split the data into train and test sets (80-20 split)
dataset_split = dataset.train_test_split(test_size=0.2)

# Access train and test splits
train_dataset = dataset_split['train']
test_dataset = dataset_split['test']

In [10]:
tokenizer = DistilBertTokenizerFast.from_pretrained(q_type_model_name)

In [ ]:
def tokenize_function(examples):
    return tokenizer(examples['question'], padding= "max_length", truncation=True)

train_dataset = train_dataset.map(tokenize_function, batched=True)
test_dataset = test_dataset.map(tokenize_function, batched=True)

# Set the format to PyTorch tensors
train_dataset.set_format(type='torch', columns=['input_ids', 'attention_mask', 'label'])
test_dataset.set_format(type='torch', columns=['input_ids', 'attention_mask', 'label'])

In [12]:
# Mapping lebel and id
id2label = {v: k for k, v in label_mapping.items()}  # Map IDs to label names
label2id = {k: v for k, v in label_mapping.items()}  # Map label names to IDs

# Retraning

In [ ]:
# Loading the pre trained model
model = DistilBertForSequenceClassification.from_pretrained(
        q_type_model_name,
        num_labels=8,
        ignore_mismatched_sizes=True,  # Allows resizing of classification head
        id2label=id2label,
        label2id=label2id,
    )
print(model.config.num_labels)  # Should print 7

In [ ]:
training_args = TrainingArguments(
    output_dir= q_type_model_result,           # Output directory
    eval_strategy="epoch",     # Evaluate after a specific number of steps
    save_strategy="epoch",           # Save the model after a specific number of steps
    learning_rate= 3e-5,
    num_train_epochs=50,             # Number of training epochs
    per_device_train_batch_size= 16,   # Batch size per device during training
    per_device_eval_batch_size= 16,    # Batch size per device during evaluation
    gradient_accumulation_steps=2,
    logging_dir='./logs',            # Directory for storing logs
    logging_steps=10,                # Log every 10 steps
    load_best_model_at_end=True,     # Required for EarlyStoppingCallback
    # use_cpu=True                     # Force CPU usage (but TrainingArguments doesn’t support this; see notes below)
)

# using default optimizer

In [ ]:
trainer = Trainer(
    model=model,                         # The model to train
    args=training_args,                  # Training arguments
    train_dataset=train_dataset,         # The training dataset
    eval_dataset=test_dataset,           # The test dataset
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
    compute_metrics=lambda p: {'accuracy': accuracy_score(p.predictions.argmax(axis=-1), p.label_ids)}  # Compute accuracy during eval
)


# Clearing memory before start traning
torch.cuda.empty_cache()
gc.collect()

# Start training
trainer.train()

### Model evaluation and Saving

In [ ]:
evaluation_results = trainer.evaluate()
evaluation_results

In [ ]:
trainer.save_model(q_type_model+"-default")
tokenizer.save_pretrained(q_type_model+"-default")

# Using AdamW optimiser with linear scheduler with warmup

In [23]:
# AdamW optimizer is automatically used by Hugging Face, but you can explicitly define it
optimizer = AdamW(model.parameters(), lr=5e-5, eps=1e-8)

# Define the linear scheduler with warmup
total_steps = len(train_dataset) * training_args.num_train_epochs // training_args.per_device_train_batch_size
warmup_steps = int(total_steps * 0.1)  # 10% of total steps for warmup
lr_scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps
)

In [ ]:
trainer = Trainer(
    model=model,                         # The model to train
    args=training_args,                  # Training arguments
    train_dataset=train_dataset,         # The training dataset
    eval_dataset=test_dataset,           # The test dataset
    optimizers=(optimizer, lr_scheduler),   # Pass optimizer and scheduler as a tuple
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
    compute_metrics=lambda p: {'accuracy': accuracy_score(p.predictions.argmax(axis=-1), p.label_ids)}  # Compute accuracy during eval
)


# Clearing memory before start traning
torch.cuda.empty_cache()
gc.collect()

# Start training
trainer.train()

In [ ]:
evaluation_results = trainer.evaluate()
evaluation_results

In [ ]:
trainer.save_model(q_type_model+"-AdamW")
tokenizer.save_pretrained(q_type_model+"-AdamW")

# Using Adam

In [ ]:
# AdamW optimizer is automatically used by Hugging Face, but you can explicitly define it
optimizer = Adam(model.parameters(), lr=3e-5)

# Define the linear scheduler with warmup
total_steps = len(train_dataset) * training_args.num_train_epochs // training_args.per_device_train_batch_size
warmup_steps = int(total_steps * 0.1)  # 10% of total steps for warmup
lr_scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps
)

In [ ]:
trainer = Trainer(
    model=model,                         # The model to train
    args=training_args,                  # Training arguments
    train_dataset=train_dataset,         # The training dataset
    eval_dataset=test_dataset,           # The test dataset
    optimizers=(optimizer, lr_scheduler),   # Pass optimizer and scheduler as a tuple
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
    compute_metrics=lambda p: {'accuracy': accuracy_score(p.predictions.argmax(axis=-1), p.label_ids)}  # Compute accuracy during eval
)


# Clearing memory before start traning
torch.cuda.empty_cache()
gc.collect()

# Start training
trainer.train()

In [ ]:
evaluation_results = trainer.evaluate()
evaluation_results

In [ ]:
trainer.save_model(q_type_model+"-Adam")
tokenizer.save_pretrained(q_type_model+"-Adam")

# Using SGD

In [ ]:
# AdamW optimizer is automatically used by Hugging Face, but you can explicitly define it
optimizer = SGD(model.parameters(), lr=0.01, momentum=0.9)

# Define the linear scheduler with warmup
total_steps = len(train_dataset) * training_args.num_train_epochs // training_args.per_device_train_batch_size
warmup_steps = int(total_steps * 0.1)  # 10% of total steps for warmup
lr_scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps
)

In [ ]:
trainer = Trainer(
    model=model,                         # The model to train
    args=training_args,                  # Training arguments
    train_dataset=train_dataset,         # The training dataset
    eval_dataset=test_dataset,           # The test dataset
    optimizers=(optimizer, lr_scheduler),   # Pass optimizer and scheduler as a tuple
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
    compute_metrics=lambda p: {'accuracy': accuracy_score(p.predictions.argmax(axis=-1), p.label_ids)}  # Compute accuracy during eval
)


# Clearing memory before start traning
torch.cuda.empty_cache()
gc.collect()

# Start training
trainer.train()

In [ ]:
evaluation_results = trainer.evaluate()
evaluation_results

In [ ]:
trainer.save_model(q_type_model+"-SGD")
tokenizer.save_pretrained(q_type_model+"-SGD")

# Using RMsprop

In [ ]:
# AdamW optimizer is automatically used by Hugging Face, but you can explicitly define it
optimizer = RMSprop(model.parameters(), lr=0.01, alpha=0.99)

# Define the linear scheduler with warmup
total_steps = len(train_dataset) * training_args.num_train_epochs // training_args.per_device_train_batch_size
warmup_steps = int(total_steps * 0.1)  # 10% of total steps for warmup
lr_scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps
)

In [ ]:
trainer = Trainer(
    model=model,                         # The model to train
    args=training_args,                  # Training arguments
    train_dataset=train_dataset,         # The training dataset
    eval_dataset=test_dataset,           # The test dataset
    optimizers=(optimizer, lr_scheduler),   # Pass optimizer and scheduler as a tuple
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
    compute_metrics=lambda p: {'accuracy': accuracy_score(p.predictions.argmax(axis=-1), p.label_ids)}  # Compute accuracy during eval
)


# Clearing memory before start traning
torch.cuda.empty_cache()
gc.collect()

# Start training
trainer.train()

In [ ]:
evaluation_results = trainer.evaluate()
evaluation_results

In [ ]:
trainer.save_model(q_type_model+"-RMSprop")
tokenizer.save_pretrained(q_type_model+"-RMSprop")